In [7]:
from presidio_analyzer import AnalyzerEngine
from presidio_anonymizer import AnonymizerEngine

# 1. Initialize the engines
analyzer = AnalyzerEngine()
anonymizer = AnonymizerEngine()

text_to_scrub = "My name is John Doe and my phone number is 212-555-5555."

# 2. Detect PII entities
analyzer_results = analyzer.analyze(text=text_to_scrub, language="en")
analyzer_results


[type: PERSON, start: 11, end: 19, score: 0.85,
 type: PHONE_NUMBER, start: 43, end: 55, score: 0.75]

In [8]:
# 3. Anonymize the text based on detection results
anonymized_result = anonymizer.anonymize(
    text=text_to_scrub, 
    analyzer_results=analyzer_results
)

print(anonymized_result.text)

My name is <PERSON> and my phone number is <PHONE_NUMBER>.


In [9]:
# 3. Define custom masking configurations per entity type
from presidio_anonymizer import OperatorConfig


operators = {
    # Replace the phone number characters with '*', but leave the last 4 visible
    "PHONE_NUMBER": OperatorConfig(
        "mask",
        {
            "masking_char": "*",
            "chars_to_mask": 8,
            "from_start": True,
            "from_end": False,
        }
    ),
    # Completely mask the email address with 'X'
    "EMAIL_ADDRESS": OperatorConfig(
        "mask",
        {
            "masking_char": "X",
            "chars_to_mask": 20,  # Mask up to 20 characters
            "from_start": True,
            "from_end": False,
        }
    ),
    # Use standard redaction/replacement for the person's name
    "PERSON": OperatorConfig("replace", {"new_value": "<REDACTED_NAME>"})
}

# 4. Anonymize the text using your configurations
anonymized_result = anonymizer.anonymize(
    text=text_to_scrub,
    analyzer_results=analyzer_results,
    operators=operators,
)

print(anonymized_result.text)


My name is <REDACTED_NAME> and my phone number is ********5555.
